In [27]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
import time, random

In [28]:
salary_cap = pd.read_csv('../../../data/external/salary_cap_history.csv')
salary_cap

,year,Salary_Cap
0,2017,94143000
1,2018,99093000
2,2019,101869000
3,2020,109140000
4,2021,109140000
5,2022,112414000
6,2023,123655000
7,2024,136021000
8,2025,140588000
9,2026,154647000


In [29]:
def Team_salary_Scrap(year):
    url = f'https://www.spotrac.com/nba/cap/_/year/{year}'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    data = pd.read_html(StringIO(response.text))[0]
    df = data.iloc[:,[1,5,6]].copy()
    df['year'] = year
    df['Team'] = df['Team'].str[:3]
    df = df.iloc[:-2,].copy()
    return df

In [30]:
years_df = []

for year in range(2017, 2025):
    team_cap_df = Team_salary_Scrap(year)
    years_df.append(team_cap_df)
    time.sleep(random.randint(1, 3))
    print(f'正在抓取 {year} 賽季 team salary cap 數據...')
print('抓取完成！')

years_df = pd.concat(years_df)
years_df

正在抓取 2017 賽季 team salary cap 數據...
正在抓取 2018 賽季 team salary cap 數據...
正在抓取 2019 賽季 team salary cap 數據...
正在抓取 2020 賽季 team salary cap 數據...
正在抓取 2021 賽季 team salary cap 數據...
正在抓取 2022 賽季 team salary cap 數據...
正在抓取 2023 賽季 team salary cap 數據...
正在抓取 2024 賽季 team salary cap 數據...
抓取完成！


,Team,Total Cap Allocations,Cap Space All,year
0,DAL,"$85,147,033","$13,945,967",2017
1,CHI,"$90,105,625","$8,987,375",2017
2,PHX,"$92,518,634","$6,574,366",2017
3,IND,"$93,661,969","$5,431,031",2017
4,ORL,"$95,538,311","$3,554,689",2017
...,...,...,...,...
25,GSW,"$199,518,342","$-58,930,342",2024
26,LAL,"$200,785,985","$-60,197,985",2024
27,WAS,"$211,677,970","$-71,089,970",2024
28,PHX,"$228,464,502","$-87,876,502",2024


In [31]:
# 建立一個將 Spotrac 縮寫轉換為 B-Ref 縮寫的字典
team_mapping = {
    'BKN': 'BRK',  # 籃網
    'CHA': 'CHO',  # 黃蜂
    'PHX': 'PHO',  # 太陽
    'NO': 'NOP',   # 鵜鶘 (有時候 Spotrac 只寫 NO)
    'NY': 'NYK',   # 尼克
    'SA': 'SAS',   # 馬刺
    'GS': 'GSW',   # 勇士
    'WSH': 'WAS'   # 巫師
}
years_df['Team'] = years_df['Team'].replace(team_mapping)
years_df

,Team,Total Cap Allocations,Cap Space All,year
0,DAL,"$85,147,033","$13,945,967",2017
1,CHI,"$90,105,625","$8,987,375",2017
2,PHO,"$92,518,634","$6,574,366",2017
3,IND,"$93,661,969","$5,431,031",2017
4,ORL,"$95,538,311","$3,554,689",2017
...,...,...,...,...
25,GSW,"$199,518,342","$-58,930,342",2024
26,LAL,"$200,785,985","$-60,197,985",2024
27,WAS,"$211,677,970","$-71,089,970",2024
28,PHO,"$228,464,502","$-87,876,502",2024


In [32]:
df = pd.merge(years_df, salary_cap, on='year', how='left')
df

,Team,Total Cap Allocations,Cap Space All,year,Salary_Cap
0,DAL,"$85,147,033","$13,945,967",2017,94143000
1,CHI,"$90,105,625","$8,987,375",2017,94143000
2,PHO,"$92,518,634","$6,574,366",2017,94143000
3,IND,"$93,661,969","$5,431,031",2017,94143000
4,ORL,"$95,538,311","$3,554,689",2017,94143000
...,...,...,...,...,...
235,GSW,"$199,518,342","$-58,930,342",2024,136021000
236,LAL,"$200,785,985","$-60,197,985",2024,136021000
237,WAS,"$211,677,970","$-71,089,970",2024,136021000
238,PHO,"$228,464,502","$-87,876,502",2024,136021000


In [33]:
# 1. 🧹 清洗資料：把 $ 和 , 拿掉，並轉換成浮點數 (float)
df['Total Cap  Allocations'] = df['Total Cap  Allocations'].replace('[\$,]', '', regex=True).astype(float)
df['Salary_Cap'] = df['Salary_Cap'].astype(float)

# 2. 🧠 特徵工程：計算我們需要的兩個核心變數
# 變數 A：球隊總薪資佔薪資帽的比例 (大於 1 代表超帽/繳豪華稅)
df['Payroll_Pct'] = df['Total Cap  Allocations'] / df['Salary_Cap']

# 變數 B：球隊剩餘薪資空間比例 (最多就是 0，不能是負數)
# 這裡直接用 1 減去 Payroll_Pct，並用 np.maximum 確保空間底線是 0
df['Cap_Space_Pct'] = np.maximum(0, 1 - df['Payroll_Pct'])

df

,Team,Total Cap Allocations,Cap Space All,year,Salary_Cap,Payroll_Pct,Cap_Space_Pct
0,DAL,85147033.0,"$13,945,967",2017,94143000.0,0.904444,0.095556
1,CHI,90105625.0,"$8,987,375",2017,94143000.0,0.957114,0.042886
2,PHO,92518634.0,"$6,574,366",2017,94143000.0,0.982746,0.017254
3,IND,93661969.0,"$5,431,031",2017,94143000.0,0.994890,0.005110
4,ORL,95538311.0,"$3,554,689",2017,94143000.0,1.014821,0.000000
...,...,...,...,...,...,...,...
235,GSW,199518342.0,"$-58,930,342",2024,136021000.0,1.466820,0.000000
236,LAL,200785985.0,"$-60,197,985",2024,136021000.0,1.476140,0.000000
237,WAS,211677970.0,"$-71,089,970",2024,136021000.0,1.556215,0.000000
238,PHO,228464502.0,"$-87,876,502",2024,136021000.0,1.679627,0.000000


In [34]:
df.to_csv('../../../data/external/team_salary_cap.csv', index=False)